# Adım 1 — Veri Hazırlığı

**Amaç:** `dataset.csv`'yi yükleyip modele verilebilir hâle getirmek:
temiz bir **X** (feature matrisi) + **y** (etiket) ve NaN'ları hallet.

Bu notebook'un sonunda: 235 koşu × 12 feature'lık `X`, iki hedef (`y_bin`, `y_multi`), ve **hiç NaN yok**.

Her hücreyi sırayla çalıştır (Shift+Enter). Önce kısa açıklama, sonra kod.

In [4]:
# --- Kütüphaneler ---
import numpy as np          # sayısal diziler (pandas'ın altyapısı)
import pandas as pd         # tablo verisi: DataFrame = satır/sütunlu tablo
from pathlib import Path    # dosya yollarını platform-bağımsız yazmak için

# --- Veri setini bul ve yükle ---
# Notebook repo kökünden de kendi klasöründen de açılabilir; iki aday yolu deniyoruz:
candidates = [
    Path("my-work/day3-4-08072026-09072026-dataset/out/dataset.csv"),   # jupyter repo kökünde açıldıysa
    Path("../day3-4-08072026-09072026-dataset/out/dataset.csv"),        # notebook klasöründe açıldıysa
]
# next(...): var olan (exists()) ilk yolu seç; hiçbiri yoksa None dön
DATA = next((p for p in candidates if p.exists()), None)
assert DATA is not None, "dataset.csv bulunamadi — jupyter'i repo kokunden veya notebook klasorunden ac"

df = pd.read_csv(DATA)       # CSV'yi DataFrame'e oku (ilk satır = kolon adları)
print("Yuklendi:", DATA)
print("Boyut (satir, kolon):", df.shape)   # .shape = (satır sayısı, kolon sayısı)

Yuklendi: ../day3-4-08072026-09072026-dataset/out/dataset.csv
Boyut (satir, kolon): (235, 18)


## 1. Veriye ilk bakış

Bir satır = bir simülasyon koşusu (**run-level**). Kolonlar üç role ayrılır:
- **metadata** — `run_id, scenario, intensity, run` (modele **girmez**; gruplama/etiket için)
- **feature** (12) — modelin bakacağı sayısal ölçümler
- **label** — `label_binary` (normal/attack), `label_class` (5 sınıf) = hedef (**y**)

In [5]:
# .head(): ilk 5 satırı tablo olarak göster — verinin neye benzediğini gör
df.head()

,run_id,scenario,intensity,run,n_flows,total_throughput_mbps,max_flow_throughput_mbps,max_flow_txpackets,flow_concentration,delivery_ratio,overall_loss_ratio,control_owd_ms,control_pdv_ms,mean_owd_ms,mean_pdv_ms,telemetry_throughput_mbps,label_class,label_binary
0,normal_i0.0_r1,normal,0.0,1,2,1.031234,0.589970,2760,0.518797,1.0,0.0,0.386805,0.131254,1.633402,0.065627,0.441263,normal,normal
1,normal_i0.0_r2,normal,0.0,2,2,1.012692,0.658793,2588,0.528055,1.0,0.0,0.413920,0.122698,1.585627,0.061349,0.353899,normal,normal
2,normal_i0.0_r3,normal,0.0,3,2,1.048468,0.629702,2697,0.519553,1.0,0.0,0.388111,0.101260,1.618056,0.050630,0.418766,normal,normal
3,normal_i0.0_r4,normal,0.0,4,2,0.920715,0.613743,2510,0.547796,1.0,0.0,0.398418,0.127402,1.612542,0.063701,0.306972,normal,normal
4,normal_i0.0_r5,normal,0.0,5,2,1.051669,0.738746,3274,0.624333,1.0,0.0,0.362657,0.103942,1.623996,0.051971,0.312923,normal,normal


In [6]:
# .info(): her kolonun adı, dolu (non-null) değer sayısı ve tipi (dtype)
# non-null < 235 olan kolon = NaN içeriyor demektir
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   run_id                     235 non-null    str    
 1   scenario                   235 non-null    str    
 2   intensity                  235 non-null    float64
 3   run                        235 non-null    int64  
 4   n_flows                    235 non-null    int64  
 5   total_throughput_mbps      235 non-null    float64
 6   max_flow_throughput_mbps   235 non-null    float64
 7   max_flow_txpackets         235 non-null    int64  
 8   flow_concentration         235 non-null    float64
 9   delivery_ratio             235 non-null    float64
 10  overall_loss_ratio         235 non-null    float64
 11  control_owd_ms             225 non-null    float64
 12  control_pdv_ms             225 non-null    float64
 13  mean_owd_ms                235 non-null    float64
 14  mean_

## 2. Sınıf dağılımı (dengeli mi?)

Model dengesiz veride çoğunluğa kayabilir Dağılımı görelim — ileride
`class_weight='balanced'` ve StratifiedKFold kararlarımızın *nedeni* budur.

In [7]:
# .value_counts(): bir kolondaki her değerin kaç kez geçtiğini sayar (büyükten küçüğe)
print("=== label_class (5 sinif) ===")
print(df["label_class"].value_counts())      # multiclass hedef: hangi sınıftan kaç koşu
print("\n=== label_binary (normal vs attack) ===")
print(df["label_binary"].value_counts())     # binary hedef: negatif sınıf (normal) ne kadar küçük

=== label_class (5 sinif) ===
label_class
greyhole     90
dos          70
normal       40
ddos         25
blackhole    10
Name: count, dtype: int64

=== label_binary (normal vs attack) ===
label_binary
attack    195
normal     40
Name: count, dtype: int64


## 3. Eksik değer (NaN) denetimi

Beklenti : **blackhole**'da yol tamamen kapalı → pump akışı yok →
`control_owd_ms` / `control_pdv_ms` ölçülemez = NaN. Bu NaN'ın kendisi bilgidir
("yol tamamen reddedilmiş"). Onaylayalım: hangi kolonda kaç NaN, hangi senaryoda?

In [8]:
# .isna() -> her hücre NaN mi (True/False); .sum() -> kolon başına NaN sayısı
nan_per_col = df.isna().sum()
print("=== NaN olan kolonlar ===")
print(nan_per_col[nan_per_col > 0])           # sadece NaN'ı olan kolonları göster

# NaN hangi senaryolarda? (beklenti: yalnız blackhole)
# .isna().any(axis=1) -> satırda (axis=1) en az bir NaN var mı
nan_rows = df[df.isna().any(axis=1)]
print("\nNaN iceren", len(nan_rows), "kosu, senaryo dagilimi:")
print(nan_rows["scenario"].value_counts())

=== NaN olan kolonlar ===
control_owd_ms    10
control_pdv_ms    10
dtype: int64

NaN iceren 10 kosu, senaryo dagilimi:
scenario
blackhole    10
Name: count, dtype: int64


## 4. X / y ayrımı

**Kural:** model yalnız X'i görür, y'yi tahmin eder. Metadata ve etiketleri
X'ten **düşeriz** — yoksa model "kopya çeker" (ör. `scenario` doğrudan cevabı verir → sahte %100).

In [9]:
# --- X'e girMEyecek kolonlar ---
META   = ["run_id", "scenario", "intensity", "run"]   # tanımlayıcı/gruplama — cevabı sızdırır
LABELS = ["label_class", "label_binary"]              # hedef (y) — X'te olamaz
# telemetry_throughput_mbps butun kosularda SABIT (zero-variance) cikti (EDA bulgusu, notebook 02).
# Sabit feature ayrim bilgisi tasimaz -> modelden cikariyoruz (CSV'de kaliyor).
DROP   = ["telemetry_throughput_mbps"]

# FEATURES = geri kalan kolonlar (liste kavraması: META+LABELS+DROP dışındakiler)
FEATURES = [c for c in df.columns if c not in META + LABELS + DROP]
print("Feature sayisi:", len(FEATURES))
print(FEATURES)

X = df[FEATURES].copy()          # feature matrisi (.copy() -> orijinal df'i bozma)
y_bin   = df["label_binary"]     # normal / attack
y_multi = df["label_class"]      # 5 sinif

print("\nX:", X.shape, "| y_bin:", y_bin.shape, "| y_multi:", y_multi.shape)
# Güvenlik: X'e metadata/label sızmadığını doğrula (leakage koruması)
# set(...) & set(...) = kesişim; boş olmalı
assert not set(META + LABELS) & set(X.columns), "X'e metadata/label sizmis!"
print("Kontrol OK: X'te metadata/label yok.")

Feature sayisi: 11
['n_flows', 'total_throughput_mbps', 'max_flow_throughput_mbps', 'max_flow_txpackets', 'flow_concentration', 'delivery_ratio', 'overall_loss_ratio', 'control_owd_ms', 'control_pdv_ms', 'mean_owd_ms', 'mean_pdv_ms']

X: (235, 11) | y_bin: (235,) | y_multi: (235,)
Kontrol OK: X'te metadata/label yok.


## 5. NaN stratejisi: impute(0) + indicator

**Karar:** NaN'ı atmıyoruz (blackhole = en şiddetli atak, kaybetmeyelim).
Yerine:
1. Bir **indicator** kolonu ekle: `control_missing` = 1 (ölçülemedi) / 0 (ölçüldü).
2. Sonra NaN'ları **0** ile doldur.

Böylece model hem sayıyı hem "eksikti" bilgisini görür. 0 burada anlamlı ("yol kapalı, teslim yok");
indicator sayesinde model "gerçekten 0" ile "ölçülemedi"yi ayırabilir. RandomForest NaN'ı doğrudan
işleyemediği için bu adım şart.

In [10]:
# 1) indicator: control_owd_ms NaN mi? .astype(int) -> True/False'i 1/0'a çevirir.
#    (control_pdv de aynı koşularda NaN olduğundan tek kolona bakmak yeter.)
X["control_missing"] = X["control_owd_ms"].isna().astype(int)

# 2) kalan NaN'ları 0 ile doldur (.fillna) — 0 = "teslim/timing yok", indicator ayrımı korur
X = X.fillna(0.0)

print("Yeni feature sayisi:", X.shape[1], "(11 + control_missing)")
print("control_missing dagilimi:")
print(X["control_missing"].value_counts())    # 1 olanlar = 10 blackhole koşusu

# RandomForest NaN işleyemez; hiç NaN kalmadığını doğrula (.sum().sum() = tüm hücrelerin toplamı)
assert X.isna().sum().sum() == 0, "Hala NaN var!"
print("\nKontrol OK: X'te hic NaN yok.")

Yeni feature sayisi: 12 (11 + control_missing)
control_missing dagilimi:
control_missing
0    225
1     10
Name: count, dtype: int64

Kontrol OK: X'te hic NaN yok.


## 6. Özet ve sonraki adım

Elimizde artık:
- `X` — 235 koşu × 12 feature, **NaN yok**
- `y_bin` — normal / attack
- `y_multi` — 5 sınıf

**Sonraki adım (Adım 3):** binary sınıflandırıcı — StratifiedKFold cross-validation ile
normal-vs-attack, F1 + confusion matrix, ve DummyClassifier baseline 

Aşağıdaki son hücre hazır X'in bir özetini verir (aklında kalsın diye).

In [11]:
# .describe() -> sayısal özet (ortalama, std, min, %25/50/75, max); .T -> transpoze (feature'lar satıra)
# min/max sütunlarına bak: ölçekler çok farklı (oran 0-1 vs paket binlerce) ->
# bu yüzden RandomForest seçtik (eşik-tabanlı; scaling gerektirmez)
X.describe().T[["mean", "min", "max"]]

,mean,min,max
n_flows,3.085106,2.000000,10.000000
total_throughput_mbps,2.160420,0.904702,9.528006
max_flow_throughput_mbps,1.207326,0.524159,8.412429
max_flow_txpackets,4690.608511,2285.000000,29000.000000
flow_concentration,0.476875,0.100393,0.863558
delivery_ratio,0.763758,0.000000,1.000000
overall_loss_ratio,0.002619,0.000000,0.100088
control_owd_ms,3.211053,0.000000,346.931378
control_pdv_ms,0.789134,0.000000,11.023311
mean_owd_ms,3.349510,1.281527,226.092209
